# 02 — Values, Types & Expressions

This notebook is the language proper, starting at the smallest piece: how Scala 3 names a value, what type it gets, and how those named values combine into expressions.

Three mental shifts to absorb here:

1. **Immutability is the default.** `val` is the everyday binding; `var` is the exception.
2. **The compiler infers types you do not write.** You can still write them — and sometimes should — but inference covers most of the surface.
3. **`if`, `while`, and `for` are expressions, not statements.** They return values. That single fact reshapes how the rest of the language is built.

## `val` vs `var`

`val` binds a name to a value once. After that, the name cannot be reassigned. `var` binds a name to a value but lets you reassign it later.

In [ ]:
val pi = 3.14159        // immutable binding
// pi = 3.0             // compile error: reassignment to val

var counter = 0         // mutable binding
counter = counter + 1   // fine

The rule of thumb in Scala is simple: **reach for `val` first**. Only switch to `var` when you have a clear reason — usually a hot loop counter or a small piece of locally mutable state. Almost all of Spark's APIs are designed around immutable data because immutability is what makes them safe to share across executors.

Two things to register about `val`:

- It is **single-assignment**, not deep-frozen. A `val` can point to a mutable collection; the *binding* cannot change, but the *thing it points to* can. Notebook 04 explores this gap in detail.
- The right-hand side is evaluated **once, eagerly**. The moment the line runs, the expression is computed and the name is bound.

## Type inference

Every binding has a static type. Most of the time you do not write it — the compiler infers it from the right-hand side.

In [ ]:
val n = 42              // inferred: Int
val pi = 3.14           // inferred: Double
val name = "ganesh"     // inferred: String
val ok = true           // inferred: Boolean

You can also write the type explicitly. The syntax is `name: Type = value`:

In [ ]:
val n: Int = 42
val pi: Double = 3.14
val name: String = "ganesh"

When **should** you write the type explicitly?

| Case | Why write the type |
|---|---|
| Public method or `val` exported from a library | Documents the contract; pins it against accidental change |
| The inferred type is wider or narrower than you want | Forces the compiler to use *your* type |
| The right-hand side is non-trivial and a reader benefits | Reads as documentation at the binding site |

Inside a method body or for short local bindings, leave inference to do its job.

## The core types

Scala's primitive numeric and Boolean types compile down to JVM primitives. The names you'll meet daily:

```
  integers:  Byte (8)   Short (16)   Int (32)   Long (64)
  floats:                              Float (32)  Double (64)
  other:     Boolean    Char (16-bit unicode)    Unit ( () )
```

- `Int` and `Double` are the defaults; literals like `42` and `3.14` infer to them.
- A `Long` literal uses an `L` suffix: `42L`. A `Float` literal uses an `f` suffix: `3.14f`.
- `String` is **not** a primitive — it is `java.lang.String`, reused directly from the JVM.
- `Char` is a 16-bit unsigned integer carrying a UTF-16 code unit. Use single quotes: `'a'`. Double quotes always mean `String`.
- `Unit` has exactly one value, written `()`. It is what a method returns when it returns nothing useful — Scala's `void`.

In [ ]:
val small: Byte = 12
val mid: Int = 1_000_000           // underscores in numeric literals are ignored
val big: Long = 9_000_000_000L
val ratio: Double = 1.0 / 3.0
val flag: Boolean = false
val letter: Char = 'g'
val nothingUseful: Unit = ()

## Type ascription

Sometimes you want to tell the compiler *exactly* which type to use for an expression — often to upcast to a parent type, or to pin a numeric width.

In [ ]:
val x = 42: Long              // type ascription: treat 42 as a Long
val y = (1, 2): (Int, Int)    // pin a tuple type

// Compare to a cast — different intent:
val z = 42.asInstanceOf[Long] // runtime cast, can fail; avoid unless needed

Ascription with `:` is **checked at compile time** — the compiler verifies the expression really is of that type (or can be widened to it). `asInstanceOf` is a runtime cast; it bypasses the compiler's check and can throw at runtime. Prefer ascription. Reach for `asInstanceOf` only when interop with Java reflection forces your hand.

## Everything is an expression

Here is the mindset shift. In most C-family languages, code is a sequence of *statements*: things you do. An `if` is a statement; it does not return anything. Same with `while`, `for`, and a block of curly braces.

In Scala, almost everything is an **expression** — code that evaluates to a value of some type. A block returns its last expression. An `if/else` returns whichever branch was taken. A `for` (when written with `yield`) returns a collection.

Statements still exist in two narrow cases:

- Assignments (`x = 1`) — these evaluate to `Unit`.
- `while` loops — these also evaluate to `Unit`.

Everything else gives you back a value you can bind, return, or pass on.

## `if` is an expression

Because `if/else` returns a value, you assign its result directly. No ternary operator needed.

In [ ]:
val n = 7
val parity = if n % 2 == 0 then "even" else "odd"
// parity: String = "odd"

Scala 3 introduced the `then` keyword in `if` expressions (the parentheses around the condition are now optional). The brace-style version still works:

In [ ]:
val parity = if (n % 2 == 0) "even" else "odd"

If you omit the `else` branch, the type of the expression becomes the **least upper bound** of the `then` branch and `Unit` — usually `Any`, which is almost never what you want. So in practice: any `if` that is supposed to *produce* a value must have an `else`.

## Block expressions

A block — anything wrapped in `{ ... }` or written with significant indentation — is itself an expression. Its value is the value of the **last** expression inside it.

In [ ]:
val area =
  val r = 5.0
  val pi = 3.14159
  pi * r * r           // <- the block's value

// area: Double = 78.53975

Two notes on blocks:

- Names bound inside a block (`r`, `pi` above) are **local** — they go out of scope at the closing brace / dedent.
- A block with no expressions has type `Unit`. A block that ends with an assignment also has type `Unit`.

## `while` is the holdout

`while` is the one structure in Scala that is purely a statement. It returns `Unit`. It exists for hot inner loops where mutation is the most efficient option, but in idiomatic Scala you rarely need it.

In [ ]:
var i = 0
while i < 3 do
  println(i)
  i += 1

// 0
// 1
// 2

Note the Scala 3 keyword `do` separating the condition from the body. Brace style without `do` still works.

Why is `while` a statement? Because to make it return a useful value, the loop would either need to accumulate something (which is what `fold` or `for ... yield` are for) or commit to mutation (which is what `var` already gives you). Scala draws the line: if you need a value back, use `for ... yield`; if you need imperative mutation, use `while`.

## `for` is an expression (preview)

A taste of what notebook 05 will cover in full. A `for` expression in Scala has two forms:

In [ ]:
// Form 1: for ... do  — runs for side effect, returns Unit
for i <- 1 to 3 do println(i)

// Form 2: for ... yield  — returns a collection
val squares = for i <- 1 to 5 yield i * i
// squares: IndexedSeq[Int] = Vector(1, 4, 9, 16, 25)

The `for ... yield` form is the one you will reach for constantly: pull elements out of a collection, transform each one, get a new collection back. Internally it is sugar over `map`, `flatMap`, and `withFilter` — notebook 05 will desugar it step by step.

## `lazy val`

One more flavour of binding worth knowing now. A `lazy val` postpones evaluation of its right-hand side until the first time the name is read. After that, the value is cached.

In [ ]:
lazy val expensive =
  println("computing...")
  (1 to 1_000_000).sum

// nothing prints yet — the right-hand side hasn't run

println(expensive)   // prints "computing..." then the sum
println(expensive)   // just prints the sum — already cached

`lazy val` is how you defer expensive work until it is actually needed, and how you break otherwise circular initialisation between values. Use it sparingly — laziness can hide where work actually happens in your program, which complicates debugging.

## Putting it together

Here is a small composition that uses everything from this notebook: a `val`, type inference, an `if` expression, and a block expression.

In [ ]:
val score = 73

val grade =
  val letter =
    if score >= 90 then "A"
    else if score >= 80 then "B"
    else if score >= 70 then "C"
    else "F"
  s"score $score => $letter"

println(grade)
// score 73 => C

Notice three things:

1. The outer `val grade = ...` binds the **value of the block**, which is the final `s"..."` interpolated string.
2. The inner `val letter = if ... else if ... else "F"` binds the **value of the if-chain**.
3. Nothing was reassigned. No `var`. No early returns. The shape is: name a value, derive the next value from it, return the last one.

## What's next

Now that you can name values and combine them into expressions, notebook 03 moves up one level: **functions and methods**. You'll see how Scala turns the lambda calculus idea of *functions as values* into something you can actually pass to `map`, return from another function, and store in a collection — the building block every subsequent notebook depends on.